#IST 664 Final Project: Email Spam Classification
**Dataset:** Enron Email Spam Corpus (Ham vs. Spam)  
**Author:** Najah Bell  
**Course:** IST 664 | Natural Language Processing
**Date:** June 2026

---
## Project Overview
This notebook implements five text classification experiments to detect spam email using the Enron corpus.  
All experiments use **5-fold cross-validation** and report **precision, recall, F1, accuracy, and confusion matrices**.

| # | Experiment | Classifier |
|---|---|---|
| 1 | Baseline: Unigrams only | Naive Bayes |
| 2 | Unigrams + Bigrams | Naive Bayes |
| 3 | Unigrams + Bigrams + POS tags | Naive Bayes |
| 4 | Full + Custom Spam Signal Features  | Naive Bayes |
| 5 | Full Feature Set | Logistic Regression |


##1:Package Instillation and Imports


In [1]:
#Install packages
import subprocess, sys
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "nltk", "scikit-learn"], check=True)

import os, sys, re, random, math, zipfile, urllib.request
from collections import Counter
import nltk
from nltk.corpus import stopwords
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import precision_recall_fscore_support, accuracy_score, confusion_matrix

#Download required NLTK data
for pkg in ['punkt', 'punkt_tab', 'stopwords', 'averaged_perceptron_tagger',
            'averaged_perceptron_tagger_eng']:
    nltk.download(pkg, quiet=True)

##2: Load the Enron Email Corpus


In [4]:
import zipfile, os, pathlib
zip_file = "FinalProjectData.zip"

#Try Google Colab upload if the file isn't already present
if not os.path.exists(zip_file):
    try:
        from google.colab import files
        print("Running in Colab — please upload FinalProjectData.zip")
        uploaded = files.upload()
        zip_file = list(uploaded.keys())[0]
    except ImportError:
        print(f"Not in Colab. Place {zip_file} next to this notebook and re-run.")

#Extract Data
extract_dir = "email_corpus"
if not os.path.exists(extract_dir):
    with zipfile.ZipFile(zip_file, 'r') as z:
        z.extractall(extract_dir)
    print(f"Extracted to {extract_dir}/")

#Find ham/spam subdirectories
corpus_dir = None
for root, dirs, files in os.walk(extract_dir):
    if 'ham' in dirs and 'spam' in dirs:
        corpus_dir = root
        break

if corpus_dir is None:
    raise FileNotFoundError("Could not find ham/ and spam/ folders inside the zip. ")

ham_dir  = os.path.join(corpus_dir, 'ham')
spam_dir = os.path.join(corpus_dir, 'spam')

n_ham  = len([f for f in os.listdir(ham_dir)  if f.endswith('.txt')])
n_spam = len([f for f in os.listdir(spam_dir) if f.endswith('.txt')])
print(f"Corpus found at: {corpus_dir}")
print(f"   Ham files : {n_ham}")
print(f"   Spam files: {n_spam}")


Running in Colab — please upload FinalProjectData.zip


Saving FinalProjectData (1).zip to FinalProjectData (1) (1).zip
Extracted to email_corpus/
Corpus found at: email_corpus/FinalProjectData/EmailSpamCorpora/corpus
   Ham files : 3672
   Spam files: 1500


## 3: Read Files & Tokenize
Reads up to `n` files from each class, tokenizes with NLTK `word_tokenize`, lowercases all tokens, and creates two document lists:
- `emaildocs` — raw lowercased tokens  
- `emaildocs_nostop` — stop-words and non-alpha tokens removed


In [5]:
n = 500
random_seed = 42
random.seed(random_seed)

stop_words = set(stopwords.words('english'))

def read_emails(folder, n):
    texts = []
    for fname in sorted(os.listdir(folder)):
        if fname.endswith('.txt') and len(texts) < n:
            with open(os.path.join(folder, fname), 'r', encoding='latin-1') as f:
                texts.append(f.read())
    return texts

hamtexts  = read_emails(ham_dir,  n)
spamtexts = read_emails(spam_dir, n)
print(f"Ham emails loaded : {len(hamtexts)}")
print(f"Spam emails loaded: {len(spamtexts)}")

#Build document lists
emaildocs_raw = []
for text in spamtexts:
    emaildocs_raw.append((nltk.word_tokenize(text), 'spam'))
for text in hamtexts:
    emaildocs_raw.append((nltk.word_tokenize(text), 'ham'))

random.shuffle(emaildocs_raw)

#Lowercase
emaildocs = [([w.lower() for w in toks], lab) for toks, lab in emaildocs_raw]

#Remove stop words and non-alpha tokens
emaildocs_nostop = [
    ([w for w in toks if w.isalpha() and w not in stop_words], lab)
    for toks, lab in emaildocs
]

print("\nSample emails (first 12 tokens):")
for doc, lab in emaildocs_nostop[:2]:
    print(f"  [{lab}]: {doc[:12]}")


Ham emails loaded : 500
Spam emails loaded: 500

Sample emails (first 12 tokens):
  [ham]: ['subject', 'fw', 'calpine', 'daily', 'gas', 'nomination', 'revised', 'original', 'message', 'ricky', 'archer', 'sent']
  [ham]: ['subject', 'duns', 'number', 'changes', 'fyi', 'forwarded', 'gary', 'l', 'payne', 'hou', 'ect', 'pm']


##4: Build Feature Vocabularies
Constructs:
- **Unigram vocabulary** — top 2,000 most frequent words  
- **Bigram vocabulary** — top 500 most frequent bigrams  
- **Spam signal lexicon** — hand-crafted domain-specific spam indicator words

In [6]:
vocab_size  = 2000
bigram_size = 500

#Unigram frequency distribution
all_words_list = [w for toks, _ in emaildocs_nostop for w in toks]
all_words = nltk.FreqDist(all_words_list)
word_features = [w for w, _ in all_words.most_common(vocab_size)]

#Bigram frequency distribution
all_bigrams_list = [bg for toks, _ in emaildocs_nostop for bg in nltk.bigrams(toks)]
all_bigrams = nltk.FreqDist(all_bigrams_list)
bigram_features = [bg for bg, _ in all_bigrams.most_common(bigram_size)]

#Self-crafted spam signal lexicon
spam_signals = {
    'free', 'win', 'winner', 'prize', 'offer', 'click', 'buy', 'money',
    'cash', 'credit', 'loan', 'earn', 'income', 'profit', 'investment',
    'discount', 'deal', 'save', 'sale', 'limited', 'urgent', 'act',
    'now', 'guarantee', 'risk', 'bonus', 'reward', 'million', 'billion',
    'dollar', 'prescription', 'pharmacy', 'drug', 'pill', 'weight',
    'diet', 'mortgage', 'refinance', 'debt', 'consolidation', 'casino',
    'poker', 'lottery', 'inheritance', 'nigeria', 'account', 'verify',
    'password', 'confirm', 'bank', 'security', 'alert', 'suspended',
    'unsubscribe', 'removal', 'remove', 'promotion'
}

LABELS = ['spam', 'ham']

print(f" Unigram vocabulary size : {len(word_features)}")
print(f"   Bigram vocabulary size  : {len(bigram_features)}")
print(f"   Spam signal lexicon size: {len(spam_signals)}")
print(f"\nTop 20 unigrams: {word_features[:20]}")


 Unigram vocabulary size : 2000
   Bigram vocabulary size  : 500
   Spam signal lexicon size: 57

Top 20 unigrams: ['ect', 'subject', 'hou', 'enron', 'please', 'com', 'e', 'meter', 'http', 'pm', 'cc', 'deal', 'gas', 'nbsp', 'get', 'need', 'know', 'td', 'thanks', 'p']


## 5: Feature Definition Functions
Four feature functions are defined, each building on the previous:
1. **Unigrams** — Boolean BOW baseline  
2. **Unigrams + Bigrams** — adds two-word sequence features  
3. **Unigrams + Bigrams + POS** — adds normalized POS tag counts  
4. **Full + Spam Signals**  — adds URL ratio, ALL-CAPS ratio, $ count, exclamation count, numeric ratio, spam lexicon count


In [7]:
#Feature Function 1: Baseline Unigrams
def document_features_unigrams(document, word_features):
    """Bag-of-words Boolean features for each word in the vocabulary."""
    document_words = set(document)
    return {'V_{}'.format(w): (w in document_words) for w in word_features}

In [8]:
#Feature Function 2: Unigrams + Bigrams
def document_features_unigrams_bigrams(document, word_features, bigram_features):
    """Adds Boolean bigram features (prefixed B_) to the unigram set."""
    document_words  = set(document)
    document_bigrams = set(nltk.bigrams(document))
    features = {'V_{}'.format(w): (w in document_words) for w in word_features}
    for bg in bigram_features:
        features['B_{}_{}'.format(bg[0], bg[1])] = (bg in document_bigrams)
    return features

In [9]:
#Feature Function 3: Unigrams + Bigrams + POS Tagging
def document_features_unigrams_bigrams_pos(document, word_features, bigram_features):
    """Adds normalized POS-tag count features for 7 informative categories."""
    document_words   = set(document)
    document_bigrams = set(nltk.bigrams(document))
    features = {'V_{}'.format(w): (w in document_words) for w in word_features}
    for bg in bigram_features:
        features['B_{}_{}'.format(bg[0], bg[1])] = (bg in document_bigrams)
    if document:
        pos_tags  = nltk.pos_tag(document[:100])
        pos_counts = Counter(tag for _, tag in pos_tags)
        total = len(pos_tags)
        for cat in ['NN', 'VB', 'JJ', 'RB', 'NNP', 'VBZ', 'JJS']:
            features['POS_{}'.format(cat)] = pos_counts.get(cat, 0) / total
    return features

In [10]:
#Feature Function 4: Full + Custom Spam Signal Features
def document_features_full(document, word_features, bigram_features, spam_signals):
    """
    Full feature set adding six stylometric spam signal features:
      SIGNAL_url_ratio        — density of URL-like tokens
      SIGNAL_caps_ratio       — ratio of ALL-CAPS tokens
      SIGNAL_has_dollar       — Boolean: any '$' present
      SIGNAL_dollar_count     — normalized count of '$'
      SIGNAL_excl_ratio       — normalized count of '!'
      SIGNAL_numeric_ratio    — ratio of tokens containing digits
      SIGNAL_spam_word_count  — normalized count of spam lexicon words
    """
    document_words   = set(document)
    document_bigrams = set(nltk.bigrams(document))
    features = {'V_{}'.format(w): (w in document_words) for w in word_features}
    for bg in bigram_features:
        features['B_{}_{}'.format(bg[0], bg[1])] = (bg in document_bigrams)
    if document:
        pos_tags   = nltk.pos_tag(document[:100])
        pos_counts = Counter(tag for _, tag in pos_tags)
        total_pos  = len(pos_tags)
        for cat in ['NN', 'VB', 'JJ', 'RB', 'NNP', 'VBZ', 'JJS']:
            features['POS_{}'.format(cat)] = pos_counts.get(cat, 0) / total_pos

    doc_text  = ' '.join(document)
    n         = max(len(document), 1)

    url_count = sum(1 for w in document
                    if 'http' in w.lower() or 'www' in w.lower() or w.lower().endswith('.com'))
    caps_count = sum(1 for w in document if w.isupper() and len(w) > 1)
    dollar_count = doc_text.count('$')
    excl_count   = doc_text.count('!')
    num_numeric  = sum(1 for w in document if re.search(r'\d', w))
    sig_count    = sum(1 for w in document if w.lower() in spam_signals)

    features['SIGNAL_url_ratio']       = url_count   / n
    features['SIGNAL_caps_ratio']      = caps_count  / n
    features['SIGNAL_has_dollar']      = dollar_count > 0
    features['SIGNAL_dollar_count']    = min(dollar_count, 10) / 10
    features['SIGNAL_excl_ratio']      = min(excl_count, 10)   / 10
    features['SIGNAL_numeric_ratio']   = num_numeric / n
    features['SIGNAL_spam_word_count'] = min(sig_count, 20)    / 20
    return features



##6: Evaluation Helper Functions
- `eval_measures()` — computes per-label precision, recall, F1  
- `print_confusion_matrix()` — prints a text confusion matrix  
- `cross_validation_PRF()` — runs k-fold CV with NLTK Naive Bayes  
- `cross_validation_sklearn()` — runs k-fold CV with Scikit-Learn Logistic Regression


In [11]:
def eval_measures(gold, predicted, labels):
    """Per-label precision, recall, F1. Returns three lists (one value per label)."""
    precision_list, recall_list, F1_list = [], [], []
    for lab in labels:
        TP = FP = FN = 0
        for g, p in zip(gold, predicted):
            if g == lab and p == lab:  TP += 1
            if g != lab and p == lab:  FP += 1
            if g == lab and p != lab:  FN += 1
        if TP == 0 or (TP + FP) == 0 or (TP + FN) == 0:
            precision_list.append(0); recall_list.append(0); F1_list.append(0)
        else:
            prec = TP / (TP + FP)
            rec  = TP / (TP + FN)
            precision_list.append(prec)
            recall_list.append(rec)
            F1_list.append(2 * prec * rec / (prec + rec))
    return precision_list, recall_list, F1_list

In [12]:
def print_confusion_matrix(gold, predicted, labels):
    print("  Confusion Matrix (rows=gold, cols=predicted):")
    print("  " + "".join(f"{l:>10}" for l in [""] + labels))
    for gl in labels:
        row = f"  {gl:>8}"
        for pl in labels:
            row += f"{sum(1 for g,p in zip(gold,predicted) if g==gl and p==pl):>10}"
        print(row)

In [13]:
def cross_validation_PRF(num_folds, featuresets, labels, name="Naive Bayes"):
    """K-fold cross-validation using NLTK NaiveBayesClassifier."""
    random.shuffle(featuresets)
    size = len(featuresets) // num_folds
    num_labels = len(labels)
    total_p = [0.0]*num_labels; total_r = [0.0]*num_labels; total_f = [0.0]*num_labels
    total_acc = 0.0
    all_gold, all_pred = [], []

    for i in range(num_folds):
        test  = featuresets[i*size:(i+1)*size]
        train = featuresets[:i*size] + featuresets[(i+1)*size:]
        clf   = nltk.NaiveBayesClassifier.train(train)
        gold  = [lab for _, lab in test]
        pred  = [clf.classify(fd) for fd, _ in test]
        all_gold.extend(gold); all_pred.extend(pred)
        acc = sum(g==p for g,p in zip(gold,pred)) / len(gold)
        total_acc += acc
        print(f"  Fold {i}: accuracy={acc:.3f}")
        p_list, r_list, f_list = eval_measures(gold, pred, labels)
        for j in range(num_labels):
            total_p[j]+=p_list[j]; total_r[j]+=r_list[j]; total_f[j]+=f_list[j]

    avg_p  = [t/num_folds for t in total_p]
    avg_r  = [t/num_folds for t in total_r]
    avg_f  = [t/num_folds for t in total_f]
    avg_acc = total_acc / num_folds

    print(f"\n  === {name} Results (5-Fold CV) ===")
    print(f"  Average Accuracy: {avg_acc:.3f}")
    print(f"  {'Label':>10}  {'Precision':>10}  {'Recall':>10}  {'F1':>10}")
    for j, lab in enumerate(labels):
        print(f"  {lab:>10}  {avg_p[j]:>10.3f}  {avg_r[j]:>10.3f}  {avg_f[j]:>10.3f}")
    macro_p = sum(avg_p)/num_labels; macro_r = sum(avg_r)/num_labels; macro_f = sum(avg_f)/num_labels
    print(f"\n  Macro Avg  P={macro_p:.3f}  R={macro_r:.3f}  F1={macro_f:.3f}")
    label_counts = Counter(lab for _,lab in featuresets)
    n = len(featuresets)
    weights = [label_counts[lab]/n for lab in labels]
    micro_p = sum(p*w for p,w in zip(avg_p,weights))
    micro_r = sum(r*w for r,w in zip(avg_r,weights))
    micro_f = sum(f*w for f,w in zip(avg_f,weights))
    print(f"  Micro Avg  P={micro_p:.3f}  R={micro_r:.3f}  F1={micro_f:.3f}")
    print_confusion_matrix(all_gold, all_pred, labels)
    return avg_acc, macro_p, macro_r, macro_f

In [14]:
def cross_validation_sklearn(num_folds, featuresets, labels, feature_keys):
    """K-fold cross-validation using Scikit-Learn Logistic Regression."""
    random.shuffle(featuresets)
    size = len(featuresets) // num_folds
    num_labels = len(labels)
    label_to_int = {lab: i for i, lab in enumerate(labels)}
    all_gold, all_pred, fold_accs = [], [], []

    def to_vec(fd):
        return [1.0 if fd.get(k) is True else float(fd.get(k, 0.0)) for k in feature_keys]

    for i in range(num_folds):
        test  = featuresets[i*size:(i+1)*size]
        train = featuresets[:i*size] + featuresets[(i+1)*size:]
        X_tr = [to_vec(fd) for fd,_ in train]; y_tr = [label_to_int[l] for _,l in train]
        X_te = [to_vec(fd) for fd,_ in test];  y_te = [label_to_int[l] for _,l in test]
        clf  = LogisticRegression(max_iter=500, solver='lbfgs')
        clf.fit(X_tr, y_tr); y_pred = clf.predict(X_te)
        all_gold.extend(y_te); all_pred.extend(y_pred)
        acc = accuracy_score(y_te, y_pred); fold_accs.append(acc)
        print(f"  Fold {i}: accuracy={acc:.3f}")

    int_labels = [label_to_int[l] for l in labels]
    p, r, f1, _ = precision_recall_fscore_support(all_gold, all_pred,
                      labels=int_labels, average=None, zero_division=0)
    avg_acc = sum(fold_accs)/num_folds
    print(f"\n  === Logistic Regression Results (5-Fold CV) ===")
    print(f"  Average Accuracy: {avg_acc:.3f}")
    print(f"  {'Label':>10}  {'Precision':>10}  {'Recall':>10}  {'F1':>10}")
    for j, lab in enumerate(labels):
        print(f"  {lab:>10}  {p[j]:>10.3f}  {r[j]:>10.3f}  {f1[j]:>10.3f}")
    print(f"\n  Macro Avg  P={p.mean():.3f}  R={r.mean():.3f}  F1={f1.mean():.3f}")
    label_counts = Counter(l for _,l in featuresets)
    n = len(featuresets)
    weights = [label_counts[l]/n for l in labels]
    print(f"  Micro Avg  P={sum(p[j]*weights[j] for j in range(num_labels)):.3f}"
          f"  R={sum(r[j]*weights[j] for j in range(num_labels)):.3f}"
          f"  F1={sum(f1[j]*weights[j] for j in range(num_labels)):.3f}")
    cm = confusion_matrix(all_gold, all_pred, labels=int_labels)
    print("  Confusion Matrix (rows=gold, cols=predicted):")
    print("  " + "".join(f"{l:>10}" for l in [""]+labels))
    for j, lab in enumerate(labels):
        print(f"  {lab:>8}" + "".join(f"{v:>10}" for v in cm[j]))
    return avg_acc, float(p.mean()), float(r.mean()), float(f1.mean())


##Experiment 1
##Baseline: Unigrams Only
**Feature set:** Boolean bag-of-words using the top 2,000 most frequent words (stop words removed).  
This is the baseline against which all other experiments are compared.


In [15]:
print("EXPERIMENT 1: Baseline — Unigram BOW")


featuresets_1 = [
    (document_features_unigrams(doc, word_features), lab)
    for doc, lab in emaildocs_nostop
]

results = {}
acc, p, r, f1 = cross_validation_PRF(5, featuresets_1, LABELS, "Naive Bayes — Unigrams")
results['Exp 1: Unigrams'] = (acc, p, r, f1)


EXPERIMENT 1: Baseline — Unigram BOW
  Fold 0: accuracy=0.975
  Fold 1: accuracy=0.960
  Fold 2: accuracy=0.970
  Fold 3: accuracy=0.975
  Fold 4: accuracy=0.970

  === Naive Bayes — Unigrams Results (5-Fold CV) ===
  Average Accuracy: 0.970
       Label   Precision      Recall          F1
        spam       0.952       0.990       0.970
         ham       0.990       0.950       0.969

  Macro Avg  P=0.971  R=0.970  F1=0.970
  Micro Avg  P=0.971  R=0.970  F1=0.970
  Confusion Matrix (rows=gold, cols=predicted):
                  spam       ham
      spam       495         5
       ham        25       475


##Experiment 2
##Unigrams + Bigrams
**Feature set:** Adds top-500 bigram features to the unigram baseline.  

Bigrams capture two-word spam phrases like *"click here"* or *"free money"* that unigrams miss.


In [16]:
print("EXPERIMENT 2: Unigrams + Bigrams")


featuresets_2 = [
    (document_features_unigrams_bigrams(doc, word_features, bigram_features), lab)
    for doc, lab in emaildocs_nostop
]

acc, p, r, f1 = cross_validation_PRF(5, featuresets_2, LABELS, "Naive Bayes — Unigrams + Bigrams")
results['Exp 2: + Bigrams'] = (acc, p, r, f1)


EXPERIMENT 2: Unigrams + Bigrams
  Fold 0: accuracy=0.960
  Fold 1: accuracy=0.965
  Fold 2: accuracy=0.960
  Fold 3: accuracy=0.925
  Fold 4: accuracy=0.960

  === Naive Bayes — Unigrams + Bigrams Results (5-Fold CV) ===
  Average Accuracy: 0.954
       Label   Precision      Recall          F1
        spam       0.917       0.998       0.955
         ham       0.998       0.911       0.952

  Macro Avg  P=0.957  R=0.955  F1=0.954
  Micro Avg  P=0.957  R=0.955  F1=0.954
  Confusion Matrix (rows=gold, cols=predicted):
                  spam       ham
      spam       499         1
       ham        45       455


##Experiment 3
##Unigrams + Bigrams + POS Tag Counts
**Feature set:** Adds normalized counts of 7 POS categories (NN, VB, JJ, RB, NNP, VBZ, JJS).  
Spam often uses imperative verbs (VB) and superlative adjectives (JJS) more than ham.


In [17]:
print("EXPERIMENT 3: Unigrams + Bigrams + POS Tags")


featuresets_3 = [
    (document_features_unigrams_bigrams_pos(doc, word_features, bigram_features), lab)
    for doc, lab in emaildocs_nostop
]

acc, p, r, f1 = cross_validation_PRF(5, featuresets_3, LABELS, "Naive Bayes — Unigrams + Bigrams + POS")
results['Exp 3: + POS Tags'] = (acc, p, r, f1)


EXPERIMENT 3: Unigrams + Bigrams + POS Tags
  Fold 0: accuracy=0.965
  Fold 1: accuracy=0.960
  Fold 2: accuracy=0.955
  Fold 3: accuracy=0.955
  Fold 4: accuracy=0.945

  === Naive Bayes — Unigrams + Bigrams + POS Results (5-Fold CV) ===
  Average Accuracy: 0.956
       Label   Precision      Recall          F1
        spam       0.921       0.998       0.958
         ham       0.998       0.913       0.953

  Macro Avg  P=0.959  R=0.956  F1=0.956
  Micro Avg  P=0.959  R=0.956  F1=0.956
  Confusion Matrix (rows=gold, cols=predicted):
                  spam       ham
      spam       499         1
       ham        43       457


## Experiment 4
##Full Feature Set with Custom Spam Signal Features
**New features:**
- `SIGNAL_url_ratio` — density of URL-like tokens
- `SIGNAL_caps_ratio` — ratio of ALL-CAPS tokens
- `SIGNAL_has_dollar` / `SIGNAL_dollar_count` — dollar-sign presence and count
- `SIGNAL_excl_ratio` — exclamation mark density
- `SIGNAL_numeric_ratio` — ratio of tokens containing digits
- `SIGNAL_spam_word_count` — count of words from hand-crafted spam lexicon


In [18]:
print("EXPERIMENT 4: Full Features + Custom Spam Signals (NOVEL)")


featuresets_4 = [
    (document_features_full(doc, word_features, bigram_features, spam_signals), lab)
    for doc, lab in emaildocs_nostop
]

acc, p, r, f1 = cross_validation_PRF(5, featuresets_4, LABELS, "Naive Bayes — Full Feature Set")
results['Exp 4: + Spam Signals'] = (acc, p, r, f1)


EXPERIMENT 4: Full Features + Custom Spam Signals (NOVEL)
  Fold 0: accuracy=0.950
  Fold 1: accuracy=0.950
  Fold 2: accuracy=0.950
  Fold 3: accuracy=0.965
  Fold 4: accuracy=0.955

  === Naive Bayes — Full Feature Set Results (5-Fold CV) ===
  Average Accuracy: 0.954
       Label   Precision      Recall          F1
        spam       0.917       0.998       0.956
         ham       0.998       0.910       0.952

  Macro Avg  P=0.958  R=0.954  F1=0.954
  Micro Avg  P=0.958  R=0.954  F1=0.954
  Confusion Matrix (rows=gold, cols=predicted):
                  spam       ham
      spam       499         1
       ham        45       455


## Experiment 5
##Logistic Regression with Full Feature Set
**Classifier:** Scikit-Learn `LogisticRegression`  
**Features:** Same as Experiment 4 (built with NLTK — no CountVectorizer or TfidfVectorizer).  
Logistic Regression is a discriminative model that handles correlated features better than Naive Bayes.


In [19]:
print("=" * 65)
print("EXPERIMENT 5 (Advanced): Logistic Regression — Full Features")
print("=" * 65)

feature_keys = list(featuresets_4[0][0].keys())
acc, p, r, f1 = cross_validation_sklearn(5, featuresets_4, LABELS, feature_keys)
results['Exp 5: Logistic Regression'] = (acc, p, r, f1)


EXPERIMENT 5 (Advanced): Logistic Regression — Full Features
  Fold 0: accuracy=0.940
  Fold 1: accuracy=0.980
  Fold 2: accuracy=0.985
  Fold 3: accuracy=0.955
  Fold 4: accuracy=0.960

  === Logistic Regression Results (5-Fold CV) ===
  Average Accuracy: 0.964
       Label   Precision      Recall          F1
        spam       0.941       0.990       0.965
         ham       0.989       0.938       0.963

  Macro Avg  P=0.965  R=0.964  F1=0.964
  Micro Avg  P=0.965  R=0.964  F1=0.964
  Confusion Matrix (rows=gold, cols=predicted):
                  spam       ham
      spam       495         5
       ham        31       469


## Results Summary
Comparison of all five experiments across accuracy, precision, recall, and F1.


In [20]:
print("RESULTS SUMMARY")
print("=" * 70)
print(f"{'Experiment':<35} {'Accuracy':>8} {'Precision':>10} {'Recall':>8} {'F1':>8}")
print("-" * 70)
for name, (acc, p, r, f1) in results.items():
    print(f"{name:<35} {acc:>8.3f} {p:>10.3f} {r:>8.3f} {f1:>8.3f}")
print("=" * 70)

print("""
Key Findings:
-Experiment 1 (Unigrams) performs best overall with the highest accuracy (0.970)
-Adding bigrams (Exp 2) reduces performance, suggesting that increasing feature
sparsity and correlation did not help Naive Bayes in this dataset.
-POS tags (Exp 3) slightly recover performance compared to bigrams alone,
but do not surpass the unigram baseline.
-Custom spam signals (Exp 4) do not improve overall performance, but maintain
stability similar to other feature-rich setups.
-ogistic Regression (Exp 5) performs competitively,
but does not exceed the unigram Naive Bayes baseline in accuracy.
""")


RESULTS SUMMARY
Experiment                          Accuracy  Precision   Recall       F1
----------------------------------------------------------------------
Exp 1: Unigrams                        0.970      0.971    0.970    0.970
Exp 2: + Bigrams                       0.954      0.957    0.955    0.954
Exp 3: + POS Tags                      0.956      0.959    0.956    0.956
Exp 4: + Spam Signals                  0.954      0.958    0.954    0.954
Exp 5: Logistic Regression             0.964      0.965    0.964    0.964

Key Findings:
-Experiment 1 (Unigrams) performs best overall with the highest accuracy (0.970)
-Adding bigrams (Exp 2) reduces performance, suggesting that increasing feature 
sparsity and correlation did not help Naive Bayes in this dataset.
-POS tags (Exp 3) slightly recover performance compared to bigrams alone,
but do not surpass the unigram baseline.
-Custom spam signals (Exp 4) do not improve overall performance, but maintain
stability similar to other featu

## Discussion & Conclusions

### Were objectives met?
Yes. All five experiments ran successfully with 5-fold cross-validation reporting accuracy, precision, recall, F1, and confusion matrices. The baseline achieved ~95–97% accuracy, and all feature additions produced competitive results.

### Training vs. Overfitting
Across all experiments, performance variance between folds is low, indicating stable generalization. No strong evidence of overfitting is observed in any model based on cross-validation results.

### Notable finding: Bigrams decreased perfomance
Unlike expectations, adding bigrams (Experiment 2) reduced performance compared to unigram-only features.

This likely happens because:

- The dataset is already highly separable using individual words.

- Adding bigrams increases sparsity significantly.

- Naive Bayes struggles with correlated and sparse feature spaces.

- Many bigrams appear too infrequently to contribute stable probability estimates.

So the drop is not surprising and reflects feature sparsity rather than model failure.

### Dataset evaluation
The Enron corpus is well-suited for this task. A larger, more recent dataset such as the SpamAssassin corpus or TREC 2007 Public Spam Corpus would provide more generalization across spam patterns (social engineering, phishing, URL shorteners).

### Alternatives that might work better
- **TF-IDF + Logistic Regression or Linear SVM** would likely outperform Naive Bayes more consistently

- **SVM (linear kernel)** is typically stronger than Naive Bayes for sparse text classification

- **Transformer-based models (e.g., BERT)** would improve contextual understanding but are unnecessary for this dataset’s simplicity

### Next steps and resources
1. Sebastiani, F. (2002). *Machine learning in automated text categorization.* ACM Computing Surveys. https://dl.acm.org/doi/10.1145/505282.505283
2. Bird, S., Klein, E., & Loper, E. (2009). *Natural Language Processing with Python.* O'Reilly. https://www.nltk.org/book/
3. Metsis, V., Androutsopoulos, I., & Paliouras, G. (2006). *Spam filtering with Naive Bayes — which Naive Bayes?* CEAS 2006. http://www.aueb.gr/users/ion/docs/ceas2006_paper.pdf
